# Optimize Iceberg
Compact tables, retain recent snapshots, and sweep unreachable files.

In [ ]:
catalog = "rekep"
catalog_properties = {}
namespace = None
branch = "root"
min_files = 2
retain = 24
snapshot_age_days = 7
orphan_age_days = 3
remove_orphans = True
metadata = True

In [ ]:
import datetime

from rekep.iceberg import IcebergCatalog

if type(retain) is not int or retain < 1:
    raise ValueError("retain must be a positive integer")
if type(min_files) is not int or min_files < 2:
    raise ValueError("min_files must be at least 2")
if snapshot_age_days is not None and snapshot_age_days < 0:
    raise ValueError("snapshot_age_days must be non-negative or null")
if orphan_age_days < 0:
    raise ValueError("orphan_age_days must be non-negative")

store = IcebergCatalog(name=catalog, properties=dict(catalog_properties))
snapshot_age = (
    None if snapshot_age_days is None else datetime.timedelta(days=snapshot_age_days)
)
orphan_age = datetime.timedelta(days=orphan_age_days)

In [ ]:
reports = {}
for dataset in store.datasets(namespace):
    reports[dataset.name] = dataset.optimize(
        branch=branch,
        min_files=min_files,
        retain=retain,
        older_than=snapshot_age,
        remove_orphans=remove_orphans,
        orphan_age=orphan_age,
        metadata=metadata,
    )

result = {
    "tables": len(reports),
    "rewritten": sum(report["rewritten"] for report in reports.values()),
    "expired": sum(report["expired"] for report in reports.values()),
    "deleted": sum(report["deleted"] for report in reports.values()),
    "bytes": sum(report["bytes"] for report in reports.values()),
    "reports": reports,
}
try:
    import scrapbook as sb
except ImportError:
    pass
else:
    sb.glue("result", result, encoder="json")
result